In [1]:
# Customer Engagement & Product Utilization Analytics for Retention Strategy
# Production-ready version with robust data handling, statistical tests, and report generation

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind
import warnings
warnings.filterwarnings('ignore')

In [3]:
# 1. DATA INGESTION & VALIDATION
# ================================
def load_and_validate(filepath='European_Bank.csv'):
    """Load dataset, validate columns, handle missing values, and perform basic checks."""
    df = pd.read_csv(filepath)

In [4]:
def load_and_validate(filepath='European_Bank.csv'):
    """Load dataset, validate columns, handle missing values, and perform basic checks."""
    df = pd.read_csv(filepath)
    
    # Check for missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f"⚠️ Missing values found:\n{missing[missing>0]}")
        df = df.dropna(subset=['Exited', 'IsActiveMember', 'NumOfProducts', 'Balance', 'EstimatedSalary'])
        print(f"Rows after dropping missing: {len(df)}")
    
    # Enforce correct data types
    df['HasCrCard'] = df['HasCrCard'].astype(int)
    df['IsActiveMember'] = df['IsActiveMember'].astype(int)
    df['Exited'] = df['Exited'].astype(int)
    
    # Outlier capping
    df['Age'] = df['Age'].clip(18, 100)
    df['CreditScore'] = df['CreditScore'].clip(300, 850)
    
    # Validate binary columns
    for col in ['HasCrCard', 'IsActiveMember', 'Exited']:
        assert set(df[col].unique()).issubset({0,1}), f"Invalid values in {col}"
    
    print("✅ Data validation passed.")
    return df

df = load_and_validate()

✅ Data validation passed.


In [5]:

# ================================
# 2. ENGAGEMENT CLASSIFICATION
# ================================
def create_engagement_profiles(df):
    """Create four meaningful engagement profiles."""
    balance_median = df['Balance'].median()
    df['Engagement_Profile'] = 'Other'
    df.loc[df['IsActiveMember'] == 1, 'Engagement_Profile'] = 'Active Engaged'
    df.loc[df['IsActiveMember'] == 0, 'Engagement_Profile'] = 'Inactive Disengaged'
    df.loc[(df['IsActiveMember'] == 1) & (df['NumOfProducts'] <= 2), 'Engagement_Profile'] = 'Active, Low Product'
    df.loc[(df['IsActiveMember'] == 0) & (df['Balance'] > balance_median), 'Engagement_Profile'] = 'Inactive, High Balance'
    return df

df = create_engagement_profiles(df)

In [6]:
# 3. STATISTICAL TESTS
# ================================
def run_statistical_tests(df):
    """Run chi-square and t-tests to validate churn differences."""
    results = {}
    # Active vs Inactive churn (categorical) -> chi-square
    cont_table = pd.crosstab(df['IsActiveMember'], df['Exited'])
    chi2, p, dof, expected = chi2_contingency(cont_table)
    results['active_vs_inactive_chi2_p'] = p
    print(f"Active vs Inactive churn – Chi-square p-value: {p:.6f}")
    
    # Product count (NumOfProducts) as numeric, compare means between churned and retained
    churned_products = df[df['Exited']==1]['NumOfProducts']
    retained_products = df[df['Exited']==0]['NumOfProducts']
    t_stat, p_ttest = ttest_ind(churned_products, retained_products, equal_var=False)
    results['product_count_ttest_p'] = p_ttest
    print(f"Product count difference – t-test p-value: {p_ttest:.6f}")
    
    return results

stats = run_statistical_tests(df)

Active vs Inactive churn – Chi-square p-value: 0.000000
Product count difference – t-test p-value: 0.000219


In [7]:
# 4. PRODUCT UTILIZATION ANALYSIS
# ================================
def product_analysis(df):
    """Churn rate by number of products and product categories."""
    prod_churn = df.groupby('NumOfProducts')['Exited'].agg(['mean', 'count']).reset_index()
    prod_churn.columns = ['NumOfProducts', 'Churn_Rate', 'Customer_Count']
    prod_churn['Churn_Rate_Pct'] = prod_churn['Churn_Rate'] * 100
    
    # Single vs multi-product
    df['Product_Category'] = np.where(df['NumOfProducts'] == 1, 'Single-Product',
                                      np.where(df['NumOfProducts'] >= 3, 'Multi-Product (3+)', '2-Products'))
    cat_churn = df.groupby('Product_Category')['Exited'].mean() * 100
    
    # Save plot
    plt.figure(figsize=(10,5))
    bars = plt.bar(prod_churn['NumOfProducts'].astype(str), prod_churn['Churn_Rate_Pct'], color='teal')
    plt.xlabel('Number of Products')
    plt.ylabel('Churn Rate (%)')
    plt.title('Churn Rate by Number of Products')
    for bar, rate in zip(bars, prod_churn['Churn_Rate_Pct']):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{rate:.1f}%', ha='center')
    plt.tight_layout()
    plt.savefig('product_churn.png', dpi=150)
    plt.close()
    
    return prod_churn, cat_churn

prod_churn, cat_churn = product_analysis(df)   



In [8]:
#5. FINANCIAL vs ENGAGEMENT ANALYSIS
def financial_engagement_analysis(df):
    """Balance quartiles vs activity, identify at-risk premium customers."""
    # Robust balance quartiles using pd.cut with percentiles
    percentiles = [0, 0.25, 0.5, 0.75, 1]
    bins = df['Balance'].quantile(percentiles).values
    # Ensure unique bin edges (if many zeros, first two quantiles might be same)
    bins = np.unique(bins)
    if len(bins) < 5:
        # If not enough unique bins, fallback to pd.qcut with try-except
        try:
            df['Balance_Quartile'] = pd.qcut(df['Balance'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')
        except ValueError:
            df['Balance_Quartile'] = pd.cut(df['Balance'], bins=4, labels=['Q1','Q2','Q3','Q4'])
    else:
        labels = ['Q1','Q2','Q3','Q4'][:len(bins)-1]
        df['Balance_Quartile'] = pd.cut(df['Balance'], bins=bins, labels=labels, include_lowest=True)
    
    # Pivot table - **FIXED TYPO: aggfunc instead of aggfununc**
    pivot = df.pivot_table(index='IsActiveMember', columns='Balance_Quartile', values='Exited', aggfunc='mean') * 100
    
    # At-risk premium: top 20% balance but inactive
    balance_80th = df['Balance'].quantile(0.8)
    at_risk = df[(df['Balance'] > balance_80th) & (df['IsActiveMember'] == 0)]
    at_risk_churn = at_risk['Exited'].mean() * 100
    
    # Save pivot as CSV
    pivot.to_csv('balance_activity_pivot.csv')
    at_risk.to_csv('at_risk_premium_customers.csv', index=False)
    
    return pivot, at_risk, at_risk_churn

In [9]:
# 6. RETENTION STRENGTH ASSESSMENT
# ================================
def retention_strength(df):
    """Define sticky profile, product depth index, and relationship strength index."""
    # Sticky customer: active, tenure>=5, products>=2
    df['Sticky_Profile'] = np.where(
        (df['IsActiveMember'] == 1) & (df['Tenure'] >= 5) & (df['NumOfProducts'] >= 2),
        'Sticky', 'Non-Sticky'
    )
    sticky_churn = df.groupby('Sticky_Profile')['Exited'].mean() * 100
    
    # Product Depth Index (relative to bank average)
    avg_prod = df['NumOfProducts'].mean()
    df['Product_Depth_Index'] = df['NumOfProducts'] / avg_prod
    
    # Relationship Strength Index (0-1)
    max_tenure = df['Tenure'].max()
    df['Tenure_Score'] = df['Tenure'] / max_tenure
    df['Activeness_Score'] = df['IsActiveMember']
    df['Product_Score'] = df['NumOfProducts'] / 4  # max products in data = 4
    df['Relationship_Strength_Index'] = (df['Tenure_Score'] + df['Activeness_Score'] + df['Product_Score']) / 3
    
    # Histogram of Relationship Strength Index
    plt.figure(figsize=(10,5))
    sns.histplot(data=df, x='Relationship_Strength_Index', hue='Exited', bins=30, alpha=0.6)
    plt.title('Relationship Strength Index Distribution by Churn Status')
    plt.xlabel('Strength Score')
    plt.tight_layout()
    plt.savefig('relationship_strength.png', dpi=150)
    plt.close()
    
    return sticky_churn, df

sticky_churn, df = retention_strength(df)


In [10]:
# 7. KPI CALCULATION & SAVE
# ================================
def calculate_kpis(df):
    """Compute all 5 KPIs and return as dictionary."""
    active_churn = df[df['IsActiveMember']==1]['Exited'].mean()
    inactive_churn = df[df['IsActiveMember']==0]['Exited'].mean()
    engagement_retention_ratio = (1 - inactive_churn) / (1 - active_churn)
    
    product_depth_avg = df['Product_Depth_Index'].mean()
    
    balance_80th = df['Balance'].quantile(0.8)
    high_balance = df[df['Balance'] > balance_80th]
    high_balance_disengagement_rate = high_balance[high_balance['IsActiveMember']==0].shape[0] / high_balance.shape[0] * 100
    
    cc_retention = df[df['HasCrCard']==1]['Exited'].mean()
    no_cc_retention = df[df['HasCrCard']==0]['Exited'].mean()
    cc_stickiness = (1 - cc_retention) / (1 - no_cc_retention)
    
    relationship_strength_avg = df['Relationship_Strength_Index'].mean()
    
    kpis = {
        'Engagement Retention Ratio': engagement_retention_ratio,
        'Product Depth Index (avg)': product_depth_avg,
        'High-Balance Disengagement Rate (%)': high_balance_disengagement_rate,
        'Credit Card Stickiness Score': cc_stickiness,
        'Relationship Strength Index (avg)': relationship_strength_avg
    }
    return kpis

kpis = calculate_kpis(df)

# Save KPIs to CSV
kpi_df = pd.DataFrame(list(kpis.items()), columns=['KPI', 'Value'])
kpi_df.to_csv('kpi_results.csv', index=False)
print("\n📊 Key Performance Indicators:")
print(kpi_df.to_string(index=False))



📊 Key Performance Indicators:
                                KPI     Value
         Engagement Retention Ratio  0.853241
          Product Depth Index (avg)  1.000000
High-Balance Disengagement Rate (%) 49.250000
       Credit Card Stickiness Score  1.007965
  Relationship Strength Index (avg)  0.466310


In [11]:
# 8. GENERATE HTML REPORT
def generate_html_report(df, prod_churn, stats, kpis, kpi_df, at_risk, at_risk_churn):
    """Create a self-contained HTML report summarizing all findings."""
    html = f"""
    <!DOCTYPE html>
    <html>
    <head><title>Bank Retention Analytics Report</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 40px; }}
        h1 {{ color: #2c3e50; }}
        .kpi {{ background: #ecf0f1; padding: 10px; margin: 10px 0; border-radius: 5px; }}
        .insight {{ background: #d5f5e3; padding: 10px; margin: 10px 0; }}
        img {{ max-width: 100%; height: auto; margin: 20px 0; }}
        table {{ border-collapse: collapse; width: 100%; }}
        th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
        th {{ background-color: #2c3e50; color: white; }}
    </style>
    </head>
    <body>
    <h1>Customer Engagement & Product Utilization Analytics</h1>
    <p><strong>Date:</strong> {pd.Timestamp.now().strftime('%Y-%m-%d')}</p>
    
    <h2>Key Performance Indicators (KPIs)</h2>
    {kpi_df.to_html(index=False)}
    
    <h2>Statistical Significance</h2>
    <ul>
        <li>Active vs Inactive churn difference – p-value: {stats['active_vs_inactive_chi2_p']:.6f} {"(Significant)" if stats['active_vs_inactive_chi2_p'] < 0.05 else "(Not significant)"}</li>
        <li>Product count difference between churned and retained – p-value: {stats['product_count_ttest_p']:.6f} {"(Significant)" if stats['product_count_ttest_p'] < 0.05 else "(Not significant)"}</li>
    </ul>
    
    <h2>Churn by Number of Products</h2>
    <img src="product_churn.png" alt="Product Churn">
    {prod_churn[['NumOfProducts','Churn_Rate_Pct','Customer_Count']].to_html(index=False)}
    
    <h2>Relationship Strength Distribution</h2>
    <img src="relationship_strength.png" alt="Relationship Strength">
    
    <h2>High-Balance Disengaged Customers</h2>
    <p><strong>Number of at-risk premium customers:</strong> {len(at_risk)}</p>
    <p><strong>Their churn rate:</strong> {at_risk_churn:.1f}% vs overall {df['Exited'].mean()*100:.1f}%</p>
    <p><em>First 10 rows saved to 'at_risk_premium_customers.csv'</em></p>
    
    <h2>Recommendations</h2>
    <div class="insight">
    <ul>
        <li>Launch targeted engagement campaigns for inactive high-balance customers.</li>
        <li>Bundle credit cards with a second product to increase stickiness (Credit Card Stickiness Score = {kpis['Credit Card Stickiness Score']:.2f}).</li>
        <li>Encourage multi-product adoption – customers with 3+ products churn at {prod_churn[prod_churn['NumOfProducts']==3]['Churn_Rate_Pct'].values[0]:.1f}% vs {prod_churn[prod_churn['NumOfProducts']==1]['Churn_Rate_Pct'].values[0]:.1f}% for single-product.</li>
        <li>Monitor Relationship Strength Index – average {kpis['Relationship Strength Index (avg)']:.2f} for retained vs {df[df['Exited']==1]['Relationship_Strength_Index'].mean():.2f} for churned.</li>
    </ul>
    </div>
    </body>
    </html>
    """
    with open('retention_report.html', 'w') as f:
        f.write(html)
    print("✅ HTML report generated: retention_report.html")